In [1]:
MNIST_MODEL_DIR = "./state_dicts/SVHN"
BATCH_SIZE = 64

In [2]:
from torch.utils.data import DataLoader, random_split
from utils.report import AverageMeter
from utils.metrics import calculate_accuracy

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')

In [3]:
SEED = 1234

# Set Seed

In [4]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# Model

In [5]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [6]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

# Load SVHN

In [7]:
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [8]:
test_transforms = transforms.Compose([
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor()])

In [9]:
download_root = '../datasets/MNIST_DATASET'
test_dataset = MNIST(download_root, transform=test_transforms, train=False, download=True)

In [10]:
len(test_dataset)

10000

# Apply Vanila LeNet in SVHN

In [11]:
def eval_loop_classifier(feature_extractor, classifier, valid_loader, loss_func, optimizer, 
                         summary_loss, summary_acc=None, device=None):
    feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            if device is not None:
                data, target = data.to(device), target.to(device)

            feature = feature_extractor(data)
            feature = torch.flatten(feature, 1)
            output = classifier(feature)

            loss = loss_func(output, target)

            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc is not None:
                acc = calculate_accuracy(output, target)
                summary_acc.update(acc, BATCH_SIZE)

    return summary_loss, summary_acc

In [12]:
def run_test(feature_extractor, classifier, device):
    feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(feature_extractor, classifier, test_loader, 
                                               criterion, None, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [13]:
feature_extractor = FeatureExtractor()
feature_extractor.load_state_dict(torch.load(os.path.join(MNIST_MODEL_DIR, "feature_extractor.pth")))

classifier = Classifier(400)
classifier.load_state_dict(torch.load(os.path.join(MNIST_MODEL_DIR, "classifier.pth")))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor.to(device)
classifier.to(device)
    
run_test(feature_extractor, classifier, device)

[test loss]1.3675412664747542 [test acc]0.5751393437385559
